
# SIMM Calculation

This dashboard demonstrates the capacity of VRE to trigger SIMM calculation from Python commands.

## Launch VRE

Kick off a process in VRE, loading all inputs from Input/vre.xml and the files referenced therein. 
This is equivalent to using the VRE command line application.

## Environment and prerequisites

- Python 3.9–3.13
- Install the self-contained VRE wheel (bundles native deps): `pip install --upgrade vannarho-risk-engine` or point pip at a local `.whl` under `wheelhouse/` or `build/wheel/`.
- No repo-built binaries or `PYTHONPATH` overrides are needed for VRE; the wheel already includes QuantLib/QuantExt/VRE libraries.


In [ ]:
# Wheel bootstrap: prefer installed wheel, fallback to local wheel or PyPI if needed
import sys
import subprocess
from pathlib import Path


def _in_repo_build(mod_path: Path) -> bool:
    parts = mod_path.resolve().parts
    return 'build' in parts and 'VREPython' in parts


def _candidate_roots():
    here = Path.cwd()
    roots = [here]
    roots.extend(list(here.parents)[:3])
    for base in list(roots):
        roots.append(base / 'wheelhouse')
        roots.append(base / 'build' / 'wheel')
    return [r for r in roots if r.exists()]


def _pick_local_wheel():
    candidates = []
    for root in _candidate_roots():
        if root.is_file() and root.suffix == '.whl':
            candidates.append(root)
        elif root.is_dir():
            candidates.extend(root.rglob('*.whl'))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


needs_install = False
existing_path = None
try:
    import VRE as _vre  # type: ignore
    existing_path = Path(_vre.__file__)
    needs_install = _in_repo_build(existing_path)
except Exception:
    needs_install = True

if needs_install:
    wheel = _pick_local_wheel()
    source = str(wheel) if wheel else 'vannarho-risk-engine'
    print(f"Installing VRE from {source} ...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', source])
else:
    print(f"VRE already available: {existing_path}")


In [ ]:
# Verify VRE comes from a wheel install (site-packages)
from pathlib import Path
import importlib.metadata as _md
import VRE as vre  # type: ignore

vre_path = Path(vre.__file__).resolve()
print('VRE module path:', vre_path)
try:
    print('vannarho-risk-engine version:', _md.version('vannarho-risk-engine'))
except _md.PackageNotFoundError:
    print('vannarho-risk-engine distribution not found (using dev build?)')

if 'build' in vre_path.parts and 'VREPython' in vre_path.parts:
    raise RuntimeError('VRE is being imported from a build tree; install the packaged wheel instead.')


In [ ]:
from VRE import *
import sys, time, math
sys.path.append('..')
import utilities

#!{sys.executable} -m pip install vannarho-risk-engine==1.8.11

params = Parameters()
params.fromFile("Input/vre_SIMM2.4_1D.xml")

vre = VREApp(params)


In [ ]:
vre.run()

utilities.checkErrorsAndRunTime(vre)

There is also the possiblity to parametrise the SIMM calculation direclty in Python

In [ ]:
# Inspecting the inputs data
inputs = vre.getInputs()

In [ ]:
crifData = "TradeID,PortfolioID,ProductClass,RiskType,Qualifier,Bucket,Label1,Label2,AmountCurrency,Amount,collect_regulations,post_regulations \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,10y,Libor3m,USD,-1991.02,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,10y,OIS,USD,-304.84,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,15y,Libor3m,USD,-611.3,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,15y,OIS,USD,-510.15,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,1y,Libor3m,USD,-0.09,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,1y,OIS,USD,-0.38,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,20y,Libor3m,USD,-5926.95,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,20y,OIS,USD,-660.82,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,2w,Libor3m,USD,11.18,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,2w,OIS,USD,-0.93,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,2y,Libor3m,USD,-0.23,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,2y,OIS,USD,-2.93,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,30y,Libor3m,USD,-1894.5,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,30y,OIS,USD,-62.5,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,3m,Libor3m,USD,0.05,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,3m,OIS,USD,-3.28,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,3y,Libor3m,USD,-1.05,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,3y,OIS,USD,-6.63,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,5y,Libor3m,USD,-1431.81,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,5y,OIS,USD,-71.68,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,6m,Libor3m,USD,-0.03,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRCurve,USD,1,6m,OIS,USD,2.01,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRVol,USD,,10y,,USD,498253.14,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRVol,USD,,15y,,USD,163454.34,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRVol,USD,,3y,,USD,2813.27,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_IRVol,USD,,5y,,USD,264121.38,ESA,USPR,SEC,CFTC \
            IR_Bermudan,CRIF_20191230,RatesFX,Risk_FX,USD,,,,USD,13186.84,ESA,USPR,SEC,CFTC "

inputs.setCrifFromBuffer(crifData)
inputs.setSimmVersion("2.13")
inputs.setSimmCalculationCurrencyCall("EUR")
inputs.setSimmCalculationCurrencyPost("EUR")
inputs.setSimmResultCurrency("EUR")
inputs.setSimmReportingCurrency("EUR")
inputs.setMporDays(1000)

# Inserting the Sensitivity analytics
inputs.insertAnalytic("SIMM")

vre.run()

utilities.checkErrorsAndRunTime(vre)

## Query Results

The results of the VRE run above have been written to the Output folder.
Moreover all results are stored in memory and can be queried as follows.

First, double-check which analytics we have requested, see Input/vre.xml:

In [ ]:
utilities.checkErrorsAndRunTime(vre)

Now list all reports that have been generated:

In [ ]:
utilities.writeList(vre.getReportNames())

In [ ]:
reportName = "marketdata"
report = vre.getReport(reportName)
utilities.checkReportStructure(reportName, report)
display(utilities.format_report(report))